XGBoost ハイパーパラメータチューニング with Snowflake Experiment Tracking

Feature Store (SALES_FORECAST_FV) から特徴量を取得し、
複数のXGBoostパラメータセットを比較評価します。

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import time
from datetime import timedelta

session = get_active_session()
session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE SCHEMA SALES_ML").collect()

In [ ]:
fs = FeatureStore(
    session=session,
    database="FOODEX_DEMO",
    name="SALES_ML",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

sales_fv = fs.get_feature_view("SALES_FORECAST_FV", "v1")
print("Feature View loaded: SALES_FORECAST_FV (v1)")

In [ ]:
features_df = fs.read_feature_view(sales_fv)

max_date = features_df.select(F.max('SALES_DATE')).collect()[0][0]
test_start_date = max_date - timedelta(days=29)

train_df = features_df.filter(F.col('SALES_DATE') < test_start_date)
test_df = features_df.filter(F.col('SALES_DATE') >= test_start_date)

print(f"Train records: {train_df.count()}, Test records: {test_df.count()}")

In [ ]:
label_encoder = LabelEncoder(
    input_cols=["CATEGORY_MEDIUM"],
    output_cols=["CATEGORY_ENCODED"]
)
label_encoder.fit(train_df)
train_encoded = label_encoder.transform(train_df)
test_encoded = label_encoder.transform(test_df)

feature_cols = ['LAG_1', 'LAG_2', 'LAG_3', 'LAG_7', 'LAG_14', 'MA_7',
                'DAY_OF_WEEK', 'IS_WEEKEND', 'MONTH_NUM', 'QUARTER_NUM',
                'TXN_COUNT', 'TOTAL_QTY', 'CATEGORY_ENCODED']
target_col = 'DAILY_SALES'

exp = ExperimentTracking(session=session)
exp.set_experiment("XGBOOST_HYPERPARAMETER_TUNING")
print("Experiment: XGBOOST_HYPERPARAMETER_TUNING")

In [ ]:
param_grid = [
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.9},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.7, "colsample_bytree": 0.7},
]

results = []

print("\n" + "="*60)
print("Starting Hyperparameter Search")
print("="*60)

In [ ]:
run_timestamp = int(time.time())

train_pd = train_encoded.to_pandas()
test_pd = test_encoded.to_pandas()

for i, params in enumerate(param_grid):
    lr_str = str(params['learning_rate']).replace('.', '_')
    run_name = f"xgb_n{params['n_estimators']}_d{params['max_depth']}_lr{lr_str}_{run_timestamp}_{i}"
    print(f"\n[{i+1}/{len(param_grid)}] {run_name}")
    
    try:
        exp.end_run()
    except:
        pass
    
    exp.start_run(run_name)
    
    full_params = {
        "n_estimators": params.get("n_estimators", 100),
        "max_depth": params.get("max_depth", 6),
        "learning_rate": params.get("learning_rate", 0.1),
        "subsample": params.get("subsample", 0.8),
        "colsample_bytree": params.get("colsample_bytree", 0.8),
        "random_state": 42
    }
    
    exp.log_params(full_params)
    exp.log_param("feature_count", len(feature_cols))
    
    from xgboost import XGBRegressor as XGBRegressorNative
    
    model = XGBRegressorNative(**full_params)
    
    start_time = time.time()
    model.fit(train_pd[feature_cols], train_pd[target_col])
    training_time = time.time() - start_time
    
    predictions = model.predict(test_pd[feature_cols])
    
    rmse = np.sqrt(mean_squared_error(test_pd[target_col], predictions))
    mape = mean_absolute_percentage_error(test_pd[target_col], predictions) * 100
    
    exp.log_metric("RMSE", rmse)
    exp.log_metric("MAPE", mape)
    exp.log_metric("training_time_sec", training_time)
    
    exp.end_run()
    
    results.append({
        "run_name": run_name,
        "params": full_params,
        "RMSE": rmse,
        "MAPE": mape,
        "training_time": training_time
    })
    
    print(f"  RMSE: {rmse:,.2f}, MAPE: {mape:.2f}%, Time: {training_time:.1f}s")

print("\n" + "="*60)
print("Results Summary")
print("="*60)

## ベストモデルをModel Registryに登録

実験トラッキングで最良と判断されたモデル（N200_D6_LR0_1）を
Snowflake Model Registryに登録し、SQL推論を可能にします。

### 登録後のメリット:
- SQLクエリから直接予測を実行可能
- バージョン管理（v1, v2...）
- 本番環境への安全なデプロイ

In [ ]:
best_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

print("Best Model Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

In [ ]:
from snowflake.ml.modeling.xgboost import XGBRegressor as SnowparkXGBRegressor

best_model = SnowparkXGBRegressor(
    input_cols=feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTED_SALES"],
    **best_params
)

print("Training best model with Snowpark ML...")
best_model.fit(train_encoded)
print("Training completed.")

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="FOODEX_DEMO",
    schema_name="SALES_ML"
)

print("Model Registry initialized.")

In [ ]:
test_pd_eval = test_encoded.to_pandas()

from xgboost import XGBRegressor as XGBRegressorNative

best_model_native = XGBRegressorNative(**best_params)
best_model_native.fit(train_pd[feature_cols], train_pd[target_col])

predictions = best_model_native.predict(test_pd_eval[feature_cols])

best_rmse = np.sqrt(mean_squared_error(test_pd_eval[target_col], predictions))
best_mape = mean_absolute_percentage_error(test_pd_eval[target_col], predictions) * 100

print(f"Best Model Metrics:")
print(f"  RMSE: {best_rmse:,.2f}")
print(f"  MAPE: {best_mape:.2f}%")

In [ ]:
sample_input = train_encoded.select(feature_cols).limit(100)

model_version = registry.log_model(
    model=best_model,
    model_name="SALES_FORECAST_MODEL",
    version_name="v2",
    sample_input_data=sample_input,
    metrics={
        "RMSE": best_rmse,
        "MAPE": best_mape,
        "n_estimators": best_params["n_estimators"],
        "max_depth": best_params["max_depth"],
        "learning_rate": best_params["learning_rate"]
    },
    comment="Best model from hyperparameter tuning experiment (N200_D6_LR0_1)"
)

print(f"\nModel registered successfully!")
print(f"  Name: {model_version.model_name}")
print(f"  Version: {model_version.version_name}")
print(f"  SQL inference enabled")

### SQL推論テスト

登録したモデルをSQLから呼び出してテストします。

In [ ]:
print("Testing inference with registered model...")
test_sample = test_pd_eval[feature_cols].head(10)
predictions_test = best_model_native.predict(test_sample)

print("\nSample Predictions:")
for i, (pred, actual) in enumerate(zip(predictions_test, test_pd_eval[target_col].head(10))):
    print(f"  [{i+1}] Predicted: {pred:,.0f}, Actual: {actual:,.0f}")

print("\n" + "="*60)
print("Model Registration Complete!")
print("="*60)
print(f"\nModel: FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL (v2)")
print(f"RMSE: {best_rmse:,.2f}, MAPE: {best_mape:.2f}%")
print(f"\nNote: For SQL inference, re-run log_model with target_platforms=['WAREHOUSE']")
print(f"\nSnowsight: AI&ML → Models → SALES_FORECAST_MODEL")

## SPCSでモデルをデプロイ（リアルタイム推論）

**SPCS (Snowpark Container Services)** にモデルをデプロイすると:
- REST APIエンドポイントとして公開
- リアルタイム推論が可能
- オートスケーリング対応

### 使用するCompute Pool:
- `SALES_FORECAST_POOL` (CPU_X64_S, 1-3 nodes)

In [ ]:
model = registry.get_model("SALES_FORECAST_MODEL")
mv = model.version("V2")

print(f"Model: {model.name}")
print(f"Version: V2")
print(f"\nDeploying to SPCS...")

In [ ]:
service = mv.create_service(
    service_name="SALES_FORECAST_SERVICE",
    service_compute_pool="SALES_FORECAST_POOL",
    image_build_compute_pool="SALES_FORECAST_POOL",
    num_workers=1,
    max_batch_rows=100
)

print(f"Service created: {service.name}")
print(f"Status: {service.status}")

In [ ]:
import time

print("Waiting for service to be ready...")
for i in range(30):
    status = session.sql(f"DESCRIBE SERVICE FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE").collect()
    current_status = status[0]['status'] if status else 'UNKNOWN'
    print(f"  [{i+1}/30] Status: {current_status}")
    if current_status == 'READY':
        print("\nService is READY!")
        break
    time.sleep(10)
else:
    print("\nService is still starting. Check status later.")

In [ ]:
print("\n" + "="*60)
print("SPCS Deployment Complete!")
print("="*60)
print(f"\nService: FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE")
print(f"Compute Pool: SALES_FORECAST_POOL")
print(f"\nTest inference:")
print("  service.predict(test_data)")
print(f"\nSnowsight: AI&ML → Models → SALES_FORECAST_MODEL → Services")